# Arbitrage-Free Valuation Framework and Option-Adjusted Spread (OAS)

This notebook details the theoretical foundations and numerical implementation of the arbitrage-free valuation framework for fixed-income securities. We address the pricing of plain vanilla and embedded-option bonds using a Binomial Interest Rate Tree, and the computation of the Option-Adjusted Spread (OAS).

## 1. The Binomial Interest Rate Tree & Backward Induction
We assume the short-term one-period forward rate $r$ evolves over discrete time steps on a recombining log-normal tree:
$$r_U = r_D \cdot e^{2\sigma}$$

Valuation is performed via **backward induction**. At any step $t$, the value is the expected discounted value of the next period's outcomes plus the coupon ($C$) received at $t$:
$$V_{t, j} = \frac{0.5 \cdot V_{t+1, U} + 0.5 \cdot V_{t+1, D} + C}{1 + r_{t, j}}$$

For bonds with **embedded options**, we enforce the option constraints before continuing the backward induction:
* **Callable:** $V_{callable} = \min(V_{computed}, K_c)$
* **Putable:** $V_{putable} = \max(V_{computed}, K_p)$

## 2. Option-Adjusted Spread (OAS)
The OAS is the constant spread added to the one-period forward rates at every node such that the model price equals the observed market price:
$$V_{node} = \frac{0.5 \cdot (V_{up} + C) + 0.5 \cdot (V_{down} + C)}{1 + r_{node} + \text{OAS}}$$

The relationship between the Z-Spread (yield spread assuming no optionality) and OAS isolates the value of the option in basis points:
$$Z\text{-Spread} = \text{OAS} + \text{Option Cost}$$


In [1]:
import numpy as np
from scipy import optimize

class BinomialInterestRateTree:
    def __init__(self, r0, sigma, periods):
        self.r0 = r0
        self.sigma = sigma
        self.periods = periods
        self.tree = self._build_tree()

    def _build_tree(self):
        tree = np.zeros((self.periods, self.periods))
        for i in range(self.periods):
            for j in range(i + 1):
                tree[j, i] = self.r0 * np.exp((2 * j - i) * self.sigma)
        return tree
    
    def display_tree(self):
        print("\nInterest Rate Tree (Rates in %):")
        for j in range(self.periods - 1, -1, -1):
            row = []
            for i in range(self.periods):
                if j <= i:
                    row.append(f"{self.tree[j, i]*100:6.2f}%")
                else:
                    row.append("       ")
            print(" | ".join(row))


In [2]:
class BondPricer:
    @staticmethod
    def price(rate_tree, face_value, coupon_rate, option_type='straight', strike=100, spread=0.0):
        """
        Prices a bond using backward induction, incorporating an optional constant spread.
        """
        periods = rate_tree.shape[1]
        coupon = face_value * coupon_rate
        
        current_values = np.full(periods + 1, face_value + coupon)
        
        for i in range(periods - 1, -1, -1):
            next_values = np.zeros(i + 1)
            for j in range(i + 1):
                # Add spread (OAS) directly to the forward rate at this node
                r = rate_tree[j, i] + spread
                
                expected_future_value = 0.5 * (current_values[j+1] + current_values[j])
                node_value = (expected_future_value + (coupon if i < periods - 1 else 0)) / (1 + r)
                
                if i != 0:
                    node_value += coupon
                
                if option_type == 'callable':
                    node_value = min(node_value, strike + (coupon if i!=0 else 0))
                elif option_type == 'putable':
                    node_value = max(node_value, strike + (coupon if i!=0 else 0))
                    
                next_values[j] = node_value
            current_values = next_values
            
        return current_values[0]

    @staticmethod
    def calculate_oas(market_price, rate_tree, face_value, coupon_rate, option_type='straight', strike=100):
        """
        Calculates the Option-Adjusted Spread (OAS) using Brent's root-finding algorithm.
        """
        # Objective function: Model Price(Spread) - Market Price = 0
        def objective(spread):
            return BondPricer.price(rate_tree, face_value, coupon_rate, option_type, strike, spread) - market_price
        
        try:
            # Search for the root within a realistic range: -1000 bps to +2000 bps
            oas = optimize.brentq(objective, -0.10, 0.20)
            return oas
        except ValueError:
            raise ValueError("OAS root not found within the specified bounds [-10%, 20%].")


In [3]:
if __name__ == "__main__":
    # Tree Parameters
    R0 = 0.05
    VOL = 0.15
    N = 5
    
    # Bond Parameters
    FACE = 100.0
    COUPON = 0.06
    CALL_PRICE = 100.0
    
    # Build Tree
    model = BinomialInterestRateTree(r0=R0, sigma=VOL, periods=N)
    
    # Scenario: A callable bond is trading in the market at $98.50.
    # What is its OAS, and what is the embedded option cost?
    observed_market_price = 98.50
    
    # 1. Calculate the Z-Spread (treat it as an option-free straight bond)
    z_spread = BondPricer.calculate_oas(observed_market_price, model.tree, FACE, COUPON, option_type='straight')
    
    # 2. Calculate the OAS (enforce the call rules on the tree)
    oas = BondPricer.calculate_oas(observed_market_price, model.tree, FACE, COUPON, option_type='callable', strike=CALL_PRICE)
    
    # 3. Derive the Option Cost
    option_cost_bps = (z_spread - oas) * 10000
    
    print("\n--- Yield Spread Analysis ---")
    print(f"Observed Market Price: ${observed_market_price:.2f}")
    print(f"Z-Spread:              {z_spread * 10000:.1f} bps")
    print(f"OAS:                   {oas * 10000:.1f} bps")
    print(f"Cost of Call Option:   {option_cost_bps:.1f} bps")
    print("\nInterpretation: The investor requires an additional", 
          f"{option_cost_bps:.1f} bps of yield purely to compensate for the risk that the issuer might call the bond.")



--- Yield Spread Analysis ---
Observed Market Price: $98.50
Z-Spread:              637.3 bps
OAS:                   636.1 bps
Cost of Call Option:   1.3 bps

Interpretation: The investor requires an additional 1.3 bps of yield purely to compensate for the risk that the issuer might call the bond.
